# Model Distillation: Amazon Nova Premier to Nova Micro

## Video Search Modality Weight Prediction

In a multimodal video search system, different queries benefit from different retrieval strategies. A query like *"red car driving"* should weight visual features heavily, while *"Cristiano Ronaldo"* should lean on metadata. Running a large model like Amazon Nova Premier for every query delivers high accuracy but at significant cost and latency.

[Amazon Bedrock Model Distillation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-distillation.html) solves this by transferring knowledge from a large **teacher** model (Nova Premier) into a small, fast **student** model (Nova Micro). The student learns to replicate the teacher's behavior from thousands of labeled examples — delivering comparable accuracy at a fraction of the cost.

### What this notebook covers

| Step | Description |
|------|-------------|
| **Data inspection** | Load and visualize 3,000 pre-labeled training examples |
| **Distillation** | Run a Bedrock distillation job (Nova Premier → Nova Micro) |
| **Deployment** | Deploy the distilled model as an on-demand endpoint |
| **Evaluation** | Compare accuracy and latency against Claude Haiku using Bedrock Model Evaluation |

### Task: Modality weight prediction

Given a video search query, predict four weights — **visual**, **audio**, **transcription**, and **metadata** — that sum to 1.0, indicating which content signals matter most for retrieval.

```
Query: "Werner discussing innovation in the city of Porto"
→ {"visual": 0.20, "audio": 0.10, "transcription": 0.45, "metadata": 0.25}
```

## 1. Setup

Install dependencies and initialize AWS clients. This notebook requires `boto3 >= 1.42.65` for the `create_custom_model_deployment` API.

**Prerequisites:** Enable [Amazon Bedrock model access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html) for Nova Premier, Nova Micro, Nova Pro, Claude Sonnet 4, and Claude Haiku 4.5.

In [ ]:
%pip install --upgrade pip boto3>=1.42.65 --quiet

In [ ]:
import json
import os
import uuid
import time
import boto3
import sagemaker
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from botocore.exceptions import ClientError
from utils import (
    create_s3_bucket,
    upload_training_data_to_s3,
    delete_s3_bucket_and_contents,
    create_model_distillation_role_and_permissions,
    delete_role_and_attached_policies,
    delete_distillation_buckets,
    add_evaluation_permissions,
)
from evaluation import (
    get_model_response,
    write_eval_dataset,
    launch_evaluation_job,
    wait_for_eval_jobs,
    parse_all_eval_results,
    benchmark_latency,
    plot_evaluation_results,
    WEIGHT_KEYS,
)

# AWS clients
bedrock_client = boto3.client(service_name="bedrock")
bedrock_runtime = boto3.client(service_name="bedrock-runtime")

# Region and account
session = boto3.session.Session()
region = session.region_name
account_id = session.client("sts").get_caller_identity()["Account"]

# S3 config
sess = sagemaker.Session()
bucket_name = sess.default_bucket()
data_prefix = "video-search-distillation-v4"

# Model identifiers
teacher_model = "us.amazon.nova-premier-v1:0"
student_model = "amazon.nova-micro-v1:0:128k"

print(f"Region:  {region}")
print(f"Account: {account_id}")
print(f"Bucket:  {bucket_name}")
print(f"Teacher: {teacher_model}")
print(f"Student: {student_model}")

## 2. System Prompts — Teacher vs Student

A key design choice in this sample is using **two different system prompts**:

| | Teacher (Nova Premier) | Student (Nova Micro) |
|--|------------------------|---------------------|
| **Purpose** | Generate high-quality labeled data | Inference at scale |
| **Length** | ~830 characters with guidelines and examples | ~120 characters, one line |
| **When used** | Offline data generation (via `generate_training_data.py`) | Baked into every training example; used at inference |

The teacher prompt includes detailed guidelines and few-shot examples to produce accurate labels. The student prompt is intentionally minimal — the model learns the task pattern from 3,000 training examples, not from verbose instructions. This **reduces input tokens by ~85%**, lowering both cost and latency at inference time.

| Modality | What it captures | Example query |
|----------|-----------------|---------------|
| **Visual** | Appearance, colors, objects, actions, scenes | "red car driving on highway" |
| **Audio** | Sounds, music, noise, non-speech audio | "thunder rumbling in background" |
| **Transcription** | Spoken words, dialogue, narration | "lecture about marine biology" |
| **Metadata** | Person names, genres, show titles, keywords | "Cristiano Ronaldo highlights" |

In [ ]:
# Teacher system prompt — detailed, used for generating labels with Nova Premier
teacher_system_message = """Analyze video search queries and assign weights (0.0-1.0) for four modalities.
Weights must sum to 1.0.

Return ONLY valid JSON in this exact format:
{
  "visual": 0.0,
  "audio": 0.0,
  "transcription": 0.0,
  "metadata": 0.0,
  "reasoning": "brief explanation"
}

Guidelines:
- visual: For appearance, colors, objects, actions, scenes, people's looks
- audio: For sounds, music, noise, non-speech audio
- transcription: For spoken words, dialogue, narration, text content
- metadata: For searching by person name, genre, captions, keywords, factual attributes

Examples:
- "red car driving" → visual=0.9, metadata=0.1
- "person saying hello" → transcription=0.5, visual=0.2, audio=0.2, metadata=0.1
- "Cristiano Ronaldo" → metadata=0.6, visual=0.3, transcription=0.1
- "dog barking loudly" → audio=0.6, visual=0.3, metadata=0.1"""

# Student system prompt — minimal, baked into distillation training data
student_system_message = """Return JSON with visual, audio, transcription, metadata weights (sum=1.0) and reasoning for the given video search query."""

print(f"Teacher prompt: {len(teacher_system_message)} chars")
print(f"Student prompt: {len(student_system_message)} chars")
print(f"Reduction: {100 - len(student_system_message)/len(teacher_system_message)*100:.0f}%")

## 3. Load Training Data

The training data was pre-generated by `generate_training_data.py`, which:
1. Generates synthetic video search queries across 5 categories (visual, audio, transcription, metadata, balanced)
2. Labels each query with Nova Premier using the detailed teacher prompt
3. Outputs results in [`bedrock-conversation-2024`](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-conversation-dataset.html) format with the short student prompt

The dataset is ready for direct upload to S3 — no format conversion needed.

In [ ]:
training_file = "distillation_dataset_v2.jsonl"

with open(training_file, "r") as f:
    training_lines = f.readlines()

print(f"Training examples: {len(training_lines)}\n")

# Show a few examples
for i in [0, 1, 5, 10]:
    record = json.loads(training_lines[i])
    query = record["messages"][0]["content"][0]["text"]
    response = json.loads(record["messages"][1]["content"][0]["text"])
    print(f"Query:   {query}")
    print(f"Weights: visual={response['visual']}, audio={response['audio']}, "
          f"transcription={response['transcription']}, metadata={response['metadata']}")
    print(f"Reason:  {response['reasoning'][:80]}...")
    print()

### 3.1 Data Analysis

Visualize the distribution of weights across the training dataset. A well-balanced dataset ensures the student model learns the full range of modality weight patterns, not just the most common ones.

In [ ]:
# Parse all completions from bedrock-conversation-2024 format
weights = {k: [] for k in WEIGHT_KEYS}

for line in training_lines:
    record = json.loads(line)
    completion = json.loads(record["messages"][1]["content"][0]["text"])
    for k in WEIGHT_KEYS:
        weights[k].append(completion.get(k, 0.0))

# Summary statistics
print("Weight Distribution Summary")
print("-" * 50)
for k in WEIGHT_KEYS:
    arr = np.array(weights[k])
    print(f"  {k:15s}  mean={arr.mean():.3f}  std={arr.std():.3f}  "
          f"min={arr.min():.1f}  max={arr.max():.1f}")

# Histograms
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]

for ax, key, color in zip(axes, WEIGHT_KEYS, colors):
    ax.hist(weights[key], bins=20, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(key.capitalize(), fontsize=13, fontweight="bold")
    ax.set_xlabel("Weight")
    ax.set_xlim(0, 1)

axes[0].set_ylabel("Count")
fig.suptitle(f"Modality Weight Distributions ({len(training_lines)} training examples)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# IAM Policy for Bedrock Model Distillation from SageMaker

 ## Overview

 This policy grants the necessary permissions for running Bedrock model customization (distillation) jobs from a SageMaker execution role. It
 includes:

 - **Bedrock permissions** for creating and managing model customization jobs
 - **IAM permissions** for creating the roles and policies required by distillation jobs

 ## Setup Instructions

 1. Navigate to the **IAM Console** → **Roles**
 2. Find your **SageMaker execution role** (e.g., `AmazonSageMaker-ExecutionRole-XXXXXXXXXX`)
 3. Under **Permissions**, click **Add permissions** → **Create inline policy**
 4. Switch to the **JSON** tab and paste the policy below
 5. Name the policy (e.g., `SageMakerBedrockDistillationAccess`)
 6. Click **Create policy**

 ## Policy

 ```json
{
      "Version": "2012-10-17",
      "Statement": [
          {
              "Sid": "BedrockFullAccess",
              "Effect": "Allow",
              "Action": "bedrock:*",
              "Resource": "*"
          },
          {
              "Sid": "IAMRoleManagementForDistillation",
              "Effect": "Allow",
              "Action": [
                  "iam:CreateRole",
                  "iam:DeleteRole",
                  "iam:GetRole",
                  "iam:PassRole",
                  "iam:AttachRolePolicy",
                  "iam:DetachRolePolicy",
                  "iam:PutRolePolicy",
                  "iam:DeleteRolePolicy",
                  "iam:UpdateAssumeRolePolicy",
                  "iam:ListAttachedRolePolicies",
                  "iam:ListRolePolicies",
                  "iam:CreatePolicy",
                  "iam:DeletePolicy",
                  "iam:GetPolicy",
                  "iam:GetPolicyVersion",
                  "iam:ListPolicyVersions",
                  "iam:DeletePolicyVersion"
              ],
              "Resource": "*"
          }
      ]
  }
 ```

 ## Notes

 - Your SageMaker execution role should already have **S3 access** for reading training data and writing output.

## 4. Upload to S3 and Configure IAM

Create the IAM role required by Bedrock distillation and upload the training data to S3. The helper functions in `utils.py` handle bucket creation, file upload, and role/policy setup.

In [ ]:
# Generate unique names for this run
job_name = f"video-search-distill-v3-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"
model_name = f"video-search-micro-v3-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

# Create IAM role for Bedrock distillation
role_name, role_arn = create_model_distillation_role_and_permissions(
    bucket_name=bucket_name, account_id=account_id
)

# Create S3 bucket (no-op if exists)
create_s3_bucket(bucket_name=bucket_name)

# Upload training data (already in bedrock-conversation-2024 format)
training_data_uri = upload_training_data_to_s3(
    bucket_name, training_file, prefix=data_prefix
)

output_path = f"s3://{bucket_name}/output/"
max_response_length = 200

print(f"\nJob name:      {job_name}")
print(f"Model name:    {model_name}")
print(f"Training data: {training_data_uri}")
print(f"Output path:   {output_path}")

## 5. Start Distillation Job

Launch a [Bedrock model distillation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-distillation.html) job. During distillation, the teacher model (Nova Premier) generates refined outputs for each training example, and the student model (Nova Micro) learns to replicate them.

> **Note:** Distillation typically takes 30-60 minutes depending on dataset size. The cell below polls for status every 60 seconds.

In [ ]:
# Wait for IAM role propagation (newly created roles need a few seconds)
print("Waiting 15 seconds for IAM role propagation...")
time.sleep(15)

response = bedrock_client.create_model_customization_job(
    jobName=job_name,
    customModelName=model_name,
    roleArn=role_arn,
    baseModelIdentifier=student_model,
    customizationType="DISTILLATION",
    trainingDataConfig={"s3Uri": training_data_uri},
    outputDataConfig={"s3Uri": output_path},
    customizationConfig={
        "distillationConfig": {
            "teacherModelConfig": {
                "teacherModelIdentifier": teacher_model,
                "maxResponseLengthForInference": max_response_length,
            }
        }
    },
)

job_arn = response["jobArn"]
print(f"Distillation job started!")
print(f"Job ARN: {job_arn}")

In [ ]:
# Monitor distillation job
while True:
    job_info = bedrock_client.get_model_customization_job(jobIdentifier=job_arn)
    status = job_info["status"]
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Status: {status}")

    if status == "Completed":
        print("\nDistillation job completed!")
        break
    elif status == "Failed":
        print(f"\nJob failed: {job_info.get('failureMessage', 'Unknown error')}")
        break
    elif status == "Stopped":
        print("\nJob was stopped.")
        break

    time.sleep(60)

## 6. Deploy Distilled Model

Create an [on-demand deployment](https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-deployment.html) for the distilled Nova Micro model. Once active, the deployment provides a model endpoint that can be called via the Bedrock Converse API.

In [ ]:
# Get custom model ARN from completed job
custom_model_arn = bedrock_client.get_model_customization_job(
    jobIdentifier=job_arn
)["outputModelArn"]

print(f"Custom Model ARN: {custom_model_arn}")

# Create on-demand deployment
deployment_name = f"video-search-v2-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

response = bedrock_client.create_custom_model_deployment(
    modelDeploymentName=deployment_name,
    modelArn=custom_model_arn,
    description="Distilled Nova Micro for video search modality weight prediction (4 weights)",
    tags=[
        {"key": "UseCase", "value": "VideoSearch"},
        {"key": "Version", "value": "v2-4weights"},
    ],
    clientRequestToken=f"deployment-{uuid.uuid4()}",
)

deployment_arn = response["customModelDeploymentArn"]
print(f"Deployment ARN:  {deployment_arn}")
print(f"Deployment Name: {deployment_name}")

In [ ]:
# Wait for deployment to become active
while True:
    status_response = bedrock_client.get_custom_model_deployment(
        customModelDeploymentIdentifier=deployment_name
    )
    status = status_response["status"]
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Status: {status}")

    if status == "Active":
        deployment_arn = status_response["customModelDeploymentArn"]
        print("\nDeployment is ready for inference!")
        break
    elif status == "Failed":
        print(f"\nFailed: {status_response.get('failureMessage', 'Unknown error')}")
        break

    time.sleep(15)

## 7. Quick Test: Distilled Micro vs Base Micro

Before running the full evaluation, do a quick sanity check. Compare the distilled Nova Micro (using the short student prompt) against the base Nova Micro (same short prompt, no distillation) on a diverse set of queries.

This shows whether distillation actually taught the model the task — the base model has never seen modality weight examples, so it should produce lower-quality or malformed outputs.

In [ ]:
# Go to AWS Console under Bedrock service and get the Distilled model ARN and add below. 
# deployment_arn = ""

custom_model_arn = ""

In [ ]:
import boto3
import uuid
import time

bedrock_client = boto3.client(service_name="bedrock", region_name="us-east-1")
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

# Step 1: Create the deployment
response = bedrock_client.create_custom_model_deployment(
  modelDeploymentName="video-search-micro-v3",
  modelArn=custom_model_arn,
  description="Distilled Nova Micro for video search intent classification",
  tags=[
      {'key': 'Project', 'value': 'video-search-nova-mme'}
  ],
  clientRequestToken=f"deployment-{uuid.uuid4()}"
)
deployment_arn = response['customModelDeploymentArn']
print(f"Deployment created: {deployment_arn}")

In [ ]:
import time

for _ in range(3):
  status = bedrock_client.get_custom_model_deployment(
      customModelDeploymentIdentifier=deployment_arn
  )['status']
  print(f"Status: {status}")
  if status == "ACTIVE":
      print("Deployment is ready!")
      break
  elif status == "FAILED":
      raise Exception("Deployment failed")
  time.sleep(10)
else:
  print(f"Deployment {status}.")

In [ ]:
# Test queries spanning all four modality types
test_queries = [
    "sunset over mountains",              # Visual
    "car chase through city streets",      # Visual
    "explosion sound effect",              # Audio
    "baby crying",                         # Audio
    "CEO discussing quarterly earnings",   # Transcription
    "interview about climate change",      # Transcription
    "Cristiano Ronaldo",                   # Metadata
    "NBA finals highlights",               # Metadata
    "2024 horror documentary",             # Metadata
    "Stranger Things season 3",            # Metadata
    "cheering crowd at concert",           # Balanced
    "politician giving speech at rally",   # Balanced
]

compare_models = [
      ("Distilled Micro", deployment_arn, student_system_message),
      ("Base Micro", "amazon.nova-micro-v1:0", student_system_message),
  ]

for query in test_queries:
  print(f"\n{'=' * 80}")
  print(f"Query: {query}")
  print(f"{'=' * 80}")
  for label, model_id, sys_msg in compare_models:
      try:
          resp = bedrock_runtime.converse(
              modelId=model_id,
              messages=[{"role": "user", "content": [{"text": query}]}],
              system=[{"text": sys_msg}],
              inferenceConfig={"maxTokens": 200, "temperature": 0.3},
          )
          raw = resp["output"]["message"]["content"][0]["text"]
          print(f"\n  [{label}]:")
          print(f"  {raw[:300]}")
      except Exception as e:
          print(f"\n  [{label}]: ERROR - {e}")

## 8. Model Evaluation

We evaluate the distilled model against Claude Haiku 4.5 as a baseline using:

1. **[Bedrock Model Evaluation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation.html)** with a custom **Overall Quality** metric (1-5 scale). Claude Sonnet judges each prediction on both weight accuracy *and* reasoning quality in a single score.
2. **Latency benchmark** — timing each model's end-to-end API response on the evaluation queries.

### 8.1 Load Evaluation Dataset

The evaluation dataset (`eval_dataset.jsonl`) contains 100 holdout samples, balanced across modality categories. Ground truth includes only the four numeric weights — no reasoning text — so the LLM judge scores purely on numeric accuracy.

In [ ]:
# Load holdout evaluation dataset
eval_file = "eval_dataset_v2.jsonl"
with open(eval_file, "r") as f:
    eval_samples = [json.loads(line) for line in f if line.strip()]

print(f"Evaluation samples: {len(eval_samples)}\n")

# Show distribution
weight_keys_check = ["visual", "audio", "transcription", "metadata"]
dom_counts = {}
for s in eval_samples:
    ref = json.loads(s["reference"])
    dom = max(weight_keys_check, key=lambda k: float(ref[k]))
    dom_counts[dom] = dom_counts.get(dom, 0) + 1
print("Distribution by dominant modality:")
for k in weight_keys_check:
    print(f"  {k:15s}: {dom_counts.get(k, 0)}")

print(f"\nSample: {eval_samples[0]['query']}")
print(f"  Ref:  {eval_samples[0]['reference']}")

### 8.2 Configure IAM and Launch Evaluation Jobs

Each model gets its own [Bedrock evaluation job](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-create.html) with a custom **OverallQuality** metric that scores both weight accuracy and reasoning quality on a 1-5 scale, judged by Claude Sonnet.

> **Note:** The distilled model uses the short student prompt; the baseline (Haiku) uses the full teacher prompt. This reflects real-world usage — the distilled model was trained to work with minimal instructions.

In [ ]:
# Add evaluation permissions to the distillation IAM role
add_evaluation_permissions(role_name, account_id)

time.sleep(10)  # Wait for policy propagation

In [ ]:
time.sleep(20)

# Models to evaluate — distilled model uses short prompt, all others use full teacher prompt
eval_models = {
    "distilled-micro": {"id": deployment_arn, "prompt": student_system_message},
    "claude-haiku": {"id": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "prompt": teacher_system_message},
}

# Judge model for custom metric evaluation
judge_model = "us.anthropic.claude-sonnet-4-20250514-v1:0"

# --- Custom Metric: Overall Quality (weights + reasoning combined) ---
# Evaluates both numeric weight accuracy AND reasoning quality in a single score.
overall_quality_metric = {
    "name": "OverallQuality",
    "instructions": (
        "You are evaluating a video search modality weight prediction model.\n\n"
        "The model predicts four weights (visual, audio, transcription, metadata summing to 1.0) "
        "and a reasoning field explaining its choices.\n\n"
        "Evaluate TWO aspects:\n"
        "A) WEIGHT ACCURACY: Compare the four numeric weights to the reference.\n"
        "B) REASONING QUALITY: Is the reasoning specific, coherent, and consistent with the weights?\n\n"
        "Steps:\n"
        "1. Extract weights from prediction and reference, compute absolute error per weight\n"
        "2. Check if the dominant modality (highest weight) matches the reference\n"
        "3. Read the reasoning — is it specific to the query or generic boilerplate?\n"
        "4. Check if reasoning is consistent with the predicted weights\n\n"
        "Query: {{prompt}}\n\n"
        "Reference weights:\n{{ground_truth}}\n\n"
        "Model prediction:\n{{prediction}}\n\n"
        "Score using the rubric considering both weight accuracy and reasoning."
    ),
    "rating_scale": [
        {"definition": "Weights within 0.05 of reference. Reasoning is specific and consistent.",
         "value": {"floatValue": 5.0}},
        {"definition": "Weights within 0.10 of reference. Reasoning is clear and mostly consistent.",
         "value": {"floatValue": 4.0}},
        {"definition": "Dominant modality matches. Avg error < 0.15. Reasoning is present but generic.",
         "value": {"floatValue": 3.0}},
        {"definition": "Dominant modality wrong OR avg error > 0.15. Reasoning vague or inconsistent.",
         "value": {"floatValue": 2.0}},
        {"definition": "Unparseable JSON, missing keys, or error > 0.30. No useful reasoning.",
         "value": {"floatValue": 1.0}},
    ],
}

custom_metrics = [overall_quality_metric]

eval_output_path = f"s3://{bucket_name}/evaluations/"

# Launch one evaluation job per model
eval_jobs = {}

for model_label, model_cfg in eval_models.items():
    # Build and upload per-model eval dataset with appropriate system prompt
    model_eval_file = f"eval_dataset_{model_label}.jsonl"
    write_eval_dataset(eval_samples, model_cfg["prompt"], model_eval_file)

    model_eval_uri = upload_training_data_to_s3(
        bucket_name, model_eval_file, prefix=f"video-search-evaluation-v3/{model_label}"
    )

    eval_job_name = f"modality-eval-{model_label}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

    try:
        job_arn = launch_evaluation_job(
            bedrock_client=bedrock_client,
            job_name=eval_job_name,
            model_id=model_cfg["id"],
            eval_data_s3_uri=model_eval_uri,
            role_arn=role_arn,
            output_s3_uri=eval_output_path,
            judge_model=judge_model,
            dataset_name=f"modality-weights-{model_label}",
            custom_metrics=custom_metrics,
        )
        eval_jobs[model_label] = job_arn
        print(f"Started eval: {model_label:20s} -> {eval_job_name}")
    except Exception as e:
        print(f"Failed to start eval for {model_label}: {e}")

    time.sleep(2)

print(f"\n{len(eval_jobs)} evaluation jobs submitted.")

In [ ]:
wait_for_eval_jobs(bedrock_client, eval_jobs)

### 8.3 Parse Evaluation Results

Download results from S3 and extract the **OverallQuality** score (1-5) for each model, covering both weight accuracy and reasoning quality.

> **Note:** Bedrock normalizes all custom metric scores to a 0-1 range. We map these back to our 1-5 rubric scale: 0.0 → 1 (Failed), 0.25 → 2, 0.5 → 3, 0.75 → 4, 1.0 → 5 (Excellent).

In [ ]:
import pandas as pd

s3_client = boto3.client("s3")
metric_names = [m["name"] for m in custom_metrics]

scores_df, summary_by_metric = parse_all_eval_results(
    bedrock_client, s3_client, bucket_name, eval_jobs, metric_names
)

# Reorder to match our preferred display order
model_order = ["distilled-micro", "claude-haiku"]
for mn in metric_names:
    summary_by_metric[mn] = summary_by_metric[mn].reindex(
        [m for m in model_order if m in summary_by_metric[mn].index]
    )

print(f"\n{'=' * 60}")
print(f"Total records: {len(scores_df)}")

for mn, summary in summary_by_metric.items():
    print(f"\n{mn} (1=Failed, 5=Excellent)")
    print("-" * 65)
    for model, row in summary.iterrows():
        print(f"  {model:20s}  mean={row['mean']:.2f}  std={row['std']:.2f}  "
              f"median={row['median']:.1f}  n={row['count']:.0f}")

### 8.4 Latency Benchmark

Bedrock Model Evaluation does not measure latency, so we benchmark it separately by timing each model's end-to-end API response on the full evaluation set. This captures real-world inference time including network overhead.

In [ ]:
latency_df = benchmark_latency(bedrock_runtime, eval_models, eval_samples)
latency_df = latency_df.reindex([m for m in model_order if m in latency_df.index])
print(f"\n{latency_df.round(0).to_string()}")

### 8.5 Results: Quality vs Latency

Side-by-side comparison of the distilled model against the baseline:
- **Overall Quality** (1-5): Combined weight accuracy + reasoning quality scored by Claude Sonnet
- **Latency**: Mean end-to-end API response time

In [ ]:
model_colors = {"distilled-micro": "#2196F3", "claude-haiku": "#FF9800"}

plot_evaluation_results(summary_by_metric, latency_df, model_order, model_colors)

## 9. Cleanup

Delete all resources created by this notebook to avoid ongoing charges. **Uncomment the lines below and run to execute cleanup.**

Resources deleted:
- On-demand model deployment (billed per inference call)
- IAM roles and attached policies
- S3 training and evaluation data

In [ ]:
# # Delete on-demand deployment
# bedrock_client.delete_custom_model_deployment(
#     customModelDeploymentIdentifier=deployment_arn
# )
# print(f"Deployment '{deployment_name}' deletion initiated")

# # Delete IAM role and policy
# delete_role_and_attached_policies(role_name=role_name)

# # Delete S3 data
# delete_distillation_buckets(bucket_name)